## Step 1: Load the Movie Ratings Data

In [15]:
import pandas as pd

cols = ["user_id", "item_id", "rating", "timestamp"]

df = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=cols
)

## Step 2: Explore the Data

In [16]:
print(df.describe())

            user_id        item_id         rating     timestamp
count  100000.00000  100000.000000  100000.000000  1.000000e+05
mean      462.48475     425.530130       3.529860  8.835289e+08
std       266.61442     330.798356       1.125674  5.343856e+06
min         1.00000       1.000000       1.000000  8.747247e+08
25%       254.00000     175.000000       3.000000  8.794487e+08
50%       447.00000     322.000000       4.000000  8.828269e+08
75%       682.00000     631.000000       4.000000  8.882600e+08
max       943.00000    1682.000000       5.000000  8.932866e+08


## Step 3: Split Each User's Ratings into a Train Set and a Test Set

In [17]:
train_list = []
test_list = []

grouped = df.groupby("user_id")

for user_id, group in grouped:
    group = group.sort_values(by="timestamp")
    
    split_index = int(0.8 * len(group))
    
    train_u = group.iloc[:split_index]
    test_u = group.iloc[split_index:]
    
    train_list.append(train_u)
    test_list.append(test_u)

train = pd.concat(train_list)
test = pd.concat(test_list)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (79619, 4)
Test shape: (20381, 4)


## Step 4: Build The Matrix Table

In [18]:
user_item_matrix = train.pivot_table(
    index="user_id",
    columns="item_id",
    values="rating"
)

In [19]:
print(user_item_matrix.shape)
user_item_matrix.head()
user_item_filled = user_item_matrix.fillna(0)
user_item_filled.head()

(943, 1613)


item_id,1,2,3,4,5,6,7,8,9,10,...,1662,1663,1664,1670,1672,1673,1675,1676,1679,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,0.0,3.0,0.0,0.0,4.0,1.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 5: Movie Similarity Matrix

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(user_item_filled.T)

## Movie Index Mapping

In [21]:
item_ids = user_item_filled.columns

## Recommendation function

In [22]:
import numpy as np

def recommend_movies(user_id, k=10):
    # Get user's ratings
    user_ratings = user_item_filled.loc[user_id]
    
    # Movies rated by user
    rated_items = user_ratings[user_ratings > 0]
    
    scores = np.zeros(len(item_ids))
    
    for item_id, rating in rated_items.items():
        item_idx = np.where(item_ids == item_id)[0][0]
        scores += item_similarity[item_idx] * rating
    
    # Remove already rated items
    for item_id in rated_items.index:
        item_idx = np.where(item_ids == item_id)[0][0]
        scores[item_idx] = 0
    
    # Get top-K indices
    top_indices = np.argsort(scores)[::-1][:k]
    
    return item_ids[top_indices]

## Test Recommendation

In [23]:
recommend_movies(1, k=10)

Index([423, 12, 100, 228, 655, 385, 568, 357, 209, 318], dtype='int64', name='item_id')

## Load Movie Titles

In [24]:
item_cols = [
    "item_id", "title", "release_date", "video_release_date",
    "IMDb_URL", "unknown", "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime", "Documentary", "Drama",
    "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Thriller", "War", "Western"
]

items = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    names=item_cols,
    encoding="latin-1"
)

items = items[["item_id", "title"]]

## Link Movie IDs to Their Names So We Can Show Titles

In [25]:
id_to_title = dict(zip(items.item_id, items.title))
title_to_id = dict(zip(items.title, items.item_id))

## Update the Function to Return Movie Names Instead of IDs

In [26]:
def recommend_movies(user_id, k=10):
    user_ratings = user_item_filled.loc[user_id]
    rated_items = user_ratings[user_ratings > 0]

    scores = np.zeros(len(item_ids))

    for item_id, rating in rated_items.items():
        item_idx = np.where(item_ids == item_id)[0][0]
        scores += item_similarity[item_idx] * rating

    for item_id in rated_items.index:
        item_idx = np.where(item_ids == item_id)[0][0]
        scores[item_idx] = 0

    top_indices = np.argsort(scores)[::-1][:k]
    recommended_ids = item_ids[top_indices]

    return [id_to_title[i] for i in recommended_ids]

## Show User History

In [27]:
def show_user_history(user_id):
    user_ratings = user_item_matrix.loc[user_id]
    watched = user_ratings[user_ratings > 0]
    
    titles = [id_to_title[i] for i in watched.index]
    return titles

## Check a User's Watch History and Get Their Recommendations

In [28]:
print(show_user_history(66))
recommend_movies(66, 10)

['Toy Story (1995)', 'Twelve Monkeys (1995)', 'Dead Man Walking (1995)', "Mr. Holland's Opus (1995)", 'Rumble in the Bronx (1995)', 'Star Wars (1977)', 'Rock, The (1996)', 'Independence Day (ID4) (1996)', 'Godfather, The (1972)', 'Return of the Jedi (1983)', 'Jerry Maguire (1996)', 'Grosse Pointe Blank (1997)', 'Men in Black (1997)', 'Contact (1997)', 'Time to Kill, A (1996)', 'Tin Cup (1996)', 'English Patient, The (1996)', 'Scream (1996)', 'Liar Liar (1997)', 'Breakdown (1997)', 'Face/Off (1997)', 'Air Force One (1997)', 'Courage Under Fire (1996)', 'Trainspotting (1996)', 'People vs. Larry Flynt, The (1996)', 'Eraser (1996)', 'Last Supper, The (1995)', 'Ransom (1996)', 'Excess Baggage (1997)', 'Con Air (1997)']


['Fargo (1996)',
 'Mission: Impossible (1996)',
 'Star Trek: First Contact (1996)',
 'Twister (1996)',
 'Truth About Cats & Dogs, The (1996)',
 'Pulp Fiction (1994)',
 'Broken Arrow (1996)',
 'Phenomenon (1996)',
 'Willy Wonka and the Chocolate Factory (1971)',
 'Silence of the Lambs, The (1991)']